# parallel_simulator_energy — analyse

Post-traitement des mesures RAPL produites par `./counter` (test **ep** — sweep $n_c$).

## Modèle SEAM

L'énergie PKG se décompose en une part **core** (indépendante de $n_c$) et une part **uncore** (partagée entre $n_c$ cœurs) :

$$E(n_c) = e_{\text{core}} + \frac{e_{\text{uncore}}}{n_c}$$

Le paramètre $S_e = e_{\text{uncore}} / e_{\text{core}}$ quantifie la **scalabilité énergétique** : plus $S_e$ est grand, plus l'efficacité énergétique s'améliore avec le nombre de cœurs.

Deux estimateurs :
- **$S_{e,\text{OLS}}$** — régression OLS sur $E_{\text{moy}}(n_c)$ pour $n_c \geq 2$
- **$\hat{S}_e(n_c)$** — **estimateur par deux points** : utilise $E(1)$ extrapolé par l'OLS et $E_{\text{med}}(n_c)$ mesuré.
  Via le greenup $G = E(1)/E(n_c)$, on obtient $\hat{S}_e = n_c(G-1)/(n_c - G)$.

**Note Barrier** : `Barrier=1` dans le CSV ne signifie pas une barrière de synchronisation réelle.
C'est la valeur minimale imposée par le générateur de benchmark pour garantir au moins un tour de boucle.
Pour `s=0`, le code de synchronisation est nul et l'overhead est négligeable.

In [ ]:
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
from scipy.stats import shapiro

# ── config ────────────────────────────────────────────────────────────────────
# CSV produit par : ./counter -i params/parasilo_ep.json -o resultats/parasilo_ep
CSV  = 'resultats/artefact_run.csv'
NODE = 'parasilo (HSW, Intel Xeon E5-2630 v3, uncore 2.4 GHz)'

COL_MAP = {
    'Energy_J':      'Energy',
    'Time_s':        'Time',
    'Temperature_C': 'Temperature',
    'Voltage_V':     'Voltage',
    'N_cores':       'N_core',
    'Nbarriers':     'Barrier',
}

## 1. Fonctions SEAM

In [ ]:
def fit_seam(df, seq_frac=0.0):
    """OLS sur E(n_c) = ecore_c + euncore_c/n_c  (n_c >= 2)."""
    sub = df[np.isclose(df['Seq_frac'], seq_frac) & (df['N_core'] >= 2)]
    ns_map = sub.groupby('N_core')['Energy'].mean()
    ns  = np.array(sorted(ns_map.index), dtype=float)
    E_m = np.array([ns_map[n] for n in ns])
    X   = np.column_stack([np.ones(len(ns)), 1.0 / ns])
    ecore_c, euncore_c = np.linalg.lstsq(X, E_m, rcond=None)[0]
    E_pred = ecore_c + euncore_c / ns
    rmse   = float(np.sqrt(np.mean((E_pred - E_m) ** 2)))
    return dict(ecore_c=ecore_c, euncore_c=euncore_c,
                Se=euncore_c / ecore_c,
                n_max=int(ns.max()), rmse=rmse, n_points=len(ns))


def compute_se_inv(df, fit, seq_frac=0.0):
    """
    Estimateur par deux points : Se_inv(n_c) = n_c*(G-1)/(n_c-G)
    avec G = E_ref / E_med(n_c) et E_ref = ecore_c + euncore_c  (E(1) du modèle OLS).
    Un point = E(1) extrapolé OLS, l'autre = E_med(n_c) mesuré.
    """
    E_ref = fit['ecore_c'] + fit['euncore_c']
    sub   = df[np.isclose(df['Seq_frac'], seq_frac) & (df['N_core'] >= 2)]
    rows  = []
    for nc, grp in sub.groupby('N_core'):
        E_med = grp['Energy'].median()
        G     = E_ref / E_med
        denom = nc - G
        if abs(denom) < 1e-6 or G <= 1:
            continue
        rows.append({'N_core': int(nc), 'E_med': E_med, 'G': G,
                     'Se_inv': nc * (G - 1) / denom})
    return pd.DataFrame(rows)


def E_seam(nc, ecore_c, euncore_c):
    return ecore_c + euncore_c / np.asarray(nc, dtype=float)


def greenup_seam(nc, ecore_c, euncore_c):
    return (ecore_c + euncore_c) / E_seam(nc, ecore_c, euncore_c)


def fit_seam_all(df):
    """Applique fit_seam pour chaque combinaison (Seq_frac, Barrier) du CSV."""
    rows = []
    for (s, b), grp in df.groupby(['Seq_frac', 'Barrier']):
        if grp['N_core'].nunique() < 2:
            continue
        try:
            f = fit_seam(grp.assign(Seq_frac=s), seq_frac=s)
            rmse_pct = f['rmse'] / grp[grp['N_core'] >= 2]['Energy'].mean() * 100
            rows.append({'Seq_frac': s, 'Barrier': int(b),
                         'Se_OLS': f['Se'], 'ecore_c': f['ecore_c'],
                         'euncore_c': f['euncore_c'], 'RMSE_pct': rmse_pct,
                         'n_max': f['n_max']})
        except Exception:
            pass
    return pd.DataFrame(rows)

## 2. Chargement

> **Rappel** : `Barrier=1` dans ce CSV ne correspond pas à une barrière de synchronisation —
> c'est la valeur minimale du compteur de tours de boucle. Pour `s=0`, l'overhead est nul.

In [ ]:
df = pd.read_csv(CSV).rename(columns=COL_MAP)
df['N_core']  = df['N_core'].astype(int)
df['Barrier'] = df['Barrier'].astype(int)

seq_fracs = sorted(df['Seq_frac'].unique())
barriers  = sorted(df['Barrier'].unique())
cores     = sorted(df['N_core'].unique())

print(f"{len(df)} mesures")
print(f"n_core   : {cores}")
print(f"Seq_frac : {seq_fracs}")
print(f"Barrier  : {barriers}  (1 = aucune synchronisation réelle, overhead nul)")
df.head()

## 3. Distributions brutes

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle(f'{NODE} — distributions brutes', fontsize=11)
for ax, col, unit in [(axes[0],'Energy','J'),(axes[1],'Time','s'),(axes[2],'Temperature','°C')]:
    sns.boxplot(data=df, x='N_core', y=col, palette='Blues', ax=ax, order=cores)
    ax.set_xlabel('Nombre de cœurs'); ax.set_ylabel(f'{col} ({unit})')
    ax.set_title(col); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 4. Régression SEAM — condition de base ($s=0$)

In [ ]:
s0 = seq_fracs[0]; b0 = barriers[0]
df0 = df[(np.isclose(df['Seq_frac'], s0)) & (df['Barrier'] == b0)]

fit = fit_seam(df0, seq_frac=s0)
rmse_pct = fit['rmse'] / df0[df0['N_core'] >= 2]['Energy'].mean() * 100

print(f"Condition : s={s0:.0%}  barrier={b0} (aucune sync réelle)")
print(f"ecore_c   = {fit['ecore_c']:.4f} J")
print(f"euncore_c = {fit['euncore_c']:.4f} J")
print(f"Se_OLS    = {fit['Se']:.4f}")
print(f"RMSE      = {fit['rmse']:.4f} J  ({rmse_pct:.2f} %)")

ns_all = sorted(df0['N_core'].unique())
E_moy  = [df0[df0['N_core'] == n]['Energy'].mean() for n in ns_all]
pos    = np.arange(len(ns_all))
E_mod  = E_seam(np.array(ns_all, dtype=float), fit['ecore_c'], fit['euncore_c'])

fig, ax = plt.subplots(figsize=(7, 4.5))
sns.boxplot(data=df0, x='N_core', y='Energy', palette='Blues', ax=ax, order=ns_all, width=0.5)
ax.plot(pos, E_moy, 'ko', ms=5, zorder=5, label='moyennes')
ax.plot(pos, E_mod, 'C1-o', lw=2, ms=5, zorder=6,
        label=f'SEAM : $e_c$={fit["ecore_c"]:.2f} J,  $e_u$={fit["euncore_c"]:.2f} J')
ax.set_xticks(pos); ax.set_xticklabels(ns_all)
ax.set_xlabel('Nombre de cœurs'); ax.set_ylabel('Énergie PKG (J)')
ax.set_title(f'{NODE}\nSEAM  $S_e$ = {fit["Se"]:.3f}  |  RMSE = {rmse_pct:.2f} %')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 5. Greenup mesuré vs modèle SEAM

In [ ]:
E_ref_med = df0[df0['N_core'] == 1]['Energy'].median()
df0 = df0.copy()
df0['greenup'] = E_ref_med / df0['Energy']

nc_arr = np.array(ns_all, dtype=float)
pos    = np.arange(len(ns_all))
G_mod  = greenup_seam(nc_arr, fit['ecore_c'], fit['euncore_c'])
G_meds = [df0[df0['N_core'] == n]['greenup'].median() for n in ns_all]
rmse_G = float(np.sqrt(np.mean([(g - greenup_seam(n, fit['ecore_c'], fit['euncore_c']))**2
                                for g, n in zip(G_meds, ns_all)]))) / np.mean(G_meds) * 100

fig, ax = plt.subplots(figsize=(7, 4.5))
sns.boxplot(data=df0, x='N_core', y='greenup', palette='Blues', ax=ax)
ax.plot(pos, G_mod, 'C1-o', lw=2, ms=5,
        label=f'SEAM  $S_e$={fit["Se"]:.3f}  RMSE {rmse_G:.1f}%%')
ax.plot(pos, nc_arr, 'k:', lw=1, label='speedup idéal')
ax.axhline(1 + fit['Se'], color='C1', ls=':', lw=1.2,
           label=f'asymptote $1+S_e$ = {1 + fit["Se"]:.3f}')
ax.set_xticks(pos); ax.set_xticklabels([int(c) for c in nc_arr])
ax.set_xlabel('Nombre de cœurs'); ax.set_ylabel('Greenup énergétique $G = E(1)/E(n)$')
ax.set_title(NODE); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 6. Cohérence $S_{e,\mathrm{OLS}}$ vs $\hat{S}_e(n_c)$ (estimateur par deux points)

$\hat{S}_e(n_c)$ est un **estimateur par deux points** : il compare $E(1)$ extrapolé par l'OLS
et $E_{\text{med}}(n_c)$ mesuré. Si le modèle est exact, $\hat{S}_e(n_c)$ doit être constant
et égal à $S_{e,\text{OLS}}$ pour tout $n_c$.

In [ ]:
inv_df   = compute_se_inv(df0, fit, seq_frac=s0)
Se_OLS   = fit['Se']
rmse_inv = float(np.sqrt(np.mean((inv_df['Se_inv'] - Se_OLS) ** 2)))

print(f"Se_OLS (régression)    = {Se_OLS:.4f}")
print(f"Se_inv médiane         = {inv_df['Se_inv'].median():.4f}")
print(f"RMSE Se_inv vs Se_OLS  = {rmse_inv:.4f}\n")
print(inv_df.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

ax = axes[0]
ax.plot(inv_df['N_core'], inv_df['Se_inv'], 'o-', color='steelblue', lw=1.8, ms=6,
        label='$\\hat{S}_e(n_c)$ (2-points)')
ax.axhline(Se_OLS, color='C1', ls='--', lw=1.8,
           label=f'$S_{{e,\\mathrm{{OLS}}}}$ = {Se_OLS:.3f}')
ax.fill_between(inv_df['N_core'], Se_OLS * 0.9, Se_OLS * 1.1,
                alpha=0.12, color='C1', label='±10 %')
ax.set_xlabel('Nombre de cœurs'); ax.set_ylabel('$\\hat{S}_e$')
ax.set_title('Estimateur par deux points par palier')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

ax = axes[1]
lim = max(Se_OLS, inv_df['Se_inv'].max()) * 1.15
ax.plot([0, lim], [0, lim], '--', color='#aaa', lw=1.2, label='y = x')
ax.fill_between([0, lim], [0, lim * 0.9], [0, lim * 1.1], color='#f0f0f0', alpha=0.6)
sc = ax.scatter(np.full(len(inv_df), Se_OLS), inv_df['Se_inv'],
                c=inv_df['N_core'], cmap='viridis', s=60,
                edgecolors='white', linewidths=0.6, zorder=3)
plt.colorbar(sc, ax=ax, label='$n_c$')
ax.text(0.05, 0.92, f'RMSE = {rmse_inv:.3f}',
        transform=ax.transAxes, fontsize=9, color='#444',
        bbox=dict(boxstyle='round,pad=0.3', fc='white', ec='#ddd', alpha=0.9))
ax.set_xlim(0, lim); ax.set_ylim(0, lim)
ax.set_xlabel('$S_{e,\\mathrm{OLS}}$'); ax.set_ylabel('$\\hat{S}_e(n_c)$')
ax.set_title('Cohérence interne des estimateurs')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

if len(inv_df) >= 3:
    stat, p_sw = shapiro(inv_df['Se_inv'])
    print(f"\nShapiro-Wilk : W={stat:.4f}  p={p_sw:.2e}  "
          f"({'normale' if p_sw > 0.05 else 'non-normale'})")

## 7. Sweep $s$ × barrières — $S_e$ par condition

Analyse toutes les combinaisons `(Seq_frac, Barrier)` présentes dans le CSV.
Avec un seul run `ep` (s=0, b=1), un seul point apparaît.
Cette cellule prend toute sa valeur avec un CSV multi-conditions (campagne complète).

In [ ]:
summary = fit_seam_all(df)
print(summary.to_string(index=False))

if len(summary) > 1:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
    ax = axes[0]
    for b, grp in summary.groupby('Barrier'):
        grp = grp.sort_values('Seq_frac')
        lbl = 'sans barrière' if b == 0 else f'{int(b)} barrière(s)'
        ax.plot(grp['Seq_frac'], grp['Se_OLS'], 'o-', label=lbl)
    ax.xaxis.set_major_formatter(ticker.PercentFormatter(xmax=1))
    ax.set_xlabel('Fraction séquentielle $s$'); ax.set_ylabel('$S_e$ (OLS)')
    ax.set_title(f'{NODE}\n$S_e$ vs fraction séquentielle')
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

    ax = axes[1]
    for s, grp in summary.groupby('Seq_frac'):
        grp = grp.sort_values('Barrier')
        ax.plot(grp['Barrier'], grp['Se_OLS'], 'o-', label=f's={s:.0%}')
    ax.set_xlabel('Nombre de barrières'); ax.set_ylabel('$S_e$ (OLS)')
    ax.set_title('$S_e$ vs nombre de barrières')
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
    plt.tight_layout(); plt.show()

    if summary['Seq_frac'].nunique() > 1 and summary['Barrier'].nunique() > 1:
        pivot = summary.pivot(index='Barrier', columns='Seq_frac', values='Se_OLS')
        pivot.index   = ['sans bar.' if b == 0 else f'{int(b)} bar.' for b in pivot.index]
        pivot.columns = [f's={v:.0%}' for v in pivot.columns]
        fig, ax = plt.subplots(figsize=(6, 3.5))
        sns.heatmap(pivot, annot=True, fmt='.2f', cmap='YlOrRd_r',
                    linewidths=0.5, ax=ax, cbar_kws={'label': '$S_e$'})
        ax.set_title(f'{NODE} — $S_e$ (OLS) par condition')
        plt.tight_layout(); plt.show()
else:
    print("(une seule condition — graphiques multi-conditions disponibles avec un CSV de campagne complète)")

## 8. Energy-Delay Product (EDP)

In [ ]:
df_edp = df0.copy()
df_edp['EDP']      = df_edp['Energy'] * df_edp['Time']
edp_ref            = df_edp[df_edp['N_core'] == 1]['EDP'].median()
df_edp['EDP_norm'] = df_edp['EDP'] / edp_ref

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle(f'{NODE} — Energy-Delay Product', fontsize=11)
sns.boxplot(data=df_edp, x='N_core', y='EDP',      palette='Purples', ax=axes[0])
axes[0].set_xlabel('Nombre de cœurs'); axes[0].set_ylabel('EDP (J·s)')
axes[0].set_title('EDP absolu'); axes[0].grid(True, alpha=0.3)
sns.boxplot(data=df_edp, x='N_core', y='EDP_norm', palette='Purples', ax=axes[1])
axes[1].axhline(1.0, color='k', ls='--', lw=0.8, label='référence n=1')
axes[1].set_xlabel('Nombre de cœurs'); axes[1].set_ylabel('EDP normalisé')
axes[1].set_title('EDP normalisé (réf. = 1 cœur)')
axes[1].legend(fontsize=8); axes[1].grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 9. Résumé

In [ ]:
rows = []
for n in sorted(df0['N_core'].unique()):
    sub   = df0[df0['N_core'] == n]
    G_m   = float(greenup_seam(n, fit['ecore_c'], fit['euncore_c']))
    inv_r = inv_df[inv_df['N_core'] == n]
    Se_i  = float(inv_r['Se_inv'].iloc[0]) if not inv_r.empty else float('nan')
    rows.append({
        'n_core':    int(n),
        'E_med (J)': round(sub['Energy'].median(), 3),
        'T_med (s)': round(sub['Time'].median(), 4),
        'Temp (°C)': round(sub['Temperature'].median(), 1),
        'G_med':     round(sub['greenup'].median(), 4) if 'greenup' in sub else '-',
        'G_SEAM':    round(G_m, 4),
        'Se_inv':    round(Se_i, 4) if np.isfinite(Se_i) else '-',
    })

result = pd.DataFrame(rows).set_index('n_core')
print(f"Node      : {NODE}")
print(f"ecore_c   = {fit['ecore_c']:.4f} J    (part fixe par run)")
print(f"euncore_c = {fit['euncore_c']:.4f} J   (part partagée entre cœurs)")
print(f"Se_OLS    = {fit['Se']:.4f}           (scalabilité énergétique)")
print(f"Asymptote G(n→∞) = 1 + Se = {1 + fit['Se']:.4f}\n")
display(result)